# AlexNet from Scratch in PyTorch

AlexNet is a **deep convolutional neural network**, which was initially developed by Alex Krizhevsky and his colleagues back in 2012. It was designed to classify images for the ImageNet LSVRC-2010 competition where it achieved state of the art results. You can read in detail about the model in the original research paper https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf.

Let's start by loading and then pre-processing the data. For our purposes, we will be using the Tiny ImageNet dataset. The Tiny ImageNetdataset consists of 64× 64 colour images from 200 object classes. Each classcontains 500 training images and 50 validation images. There are 100,000 images for training and 10,000 images as the test set for evaluation.
![Tiny ImageNet 200](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSgkClK5OiqxgtIVbBlcPgq0956i0qWayGcMA&s)

## Importing Libraries

The Notebook knows to use a GPU to train the model if it's available.

In [3]:
import numpy as np

import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler
from datetime import datetime

# Device configuration
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print('device:', device)

device: cpu


## Loading the Tiny ImageNet Dataset

Using torchvision (a helper library for computer vision tasks), we will load our dataset. This method has some helper functions that makes pre-processing pretty easy and straight-forward. Let's define the functions get_train_valid_loader and get_test_loader, and then call them to load in and process our Tiny ImageNet data:

In [4]:
import shutil
import urllib.request
import zipfile
import numpy as np
from pathlib import Path
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import datasets, transforms

# ── Data preparation helpers ──────────────────────────────────────────────────

_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"


def _download_and_extract(data_dir: Path) -> Path:
    zip_path = data_dir / "tiny-imagenet-200.zip"
    root = data_dir / "tiny-imagenet-200"
    data_dir.mkdir(parents=True, exist_ok=True)

    if not zip_path.exists():
        print("Downloading Tiny ImageNet...")
        urllib.request.urlretrieve(_URL, zip_path)

    if not root.exists():
        print("Extracting Tiny ImageNet...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(data_dir)

    return root


def _reorganize_val(root: Path) -> None:
    """Copy val images into per-class subdirs so ImageFolder can read them."""
    val_by_class = root / "val_by_class"
    if val_by_class.exists():
        return

    print("Preparing val_by_class for ImageFolder...")
    val_by_class.mkdir()
    ann_file = root / "val" / "val_annotations.txt"
    val_img_dir = root / "val" / "images"

    with open(ann_file) as f:
        for line in f:
            img, cls = line.split()[:2]
            dst = val_by_class / cls
            dst.mkdir(exist_ok=True)
            shutil.copy2(val_img_dir / img, dst / img)


def prepare_tiny_imagenet_data(
    source: str = "local",
    local_dir: str = "./dataset-tiny-imagenet-200",
    data_dir: str = "./data",
) -> Path:
    """Return the dataset root, downloading/extracting if necessary.

    source="local"  — use pre-extracted data at local_dir (no download).
    source="remote" — download from Stanford and extract into data_dir.
    """
    if source == "local":
        root = Path(local_dir).resolve()
        if not root.exists():
            raise FileNotFoundError(
                f"Local dataset not found: {root}\n"
                "Extract tiny-imagenet-200 manually or use source='remote'."
            )
        print(f"Using local dataset: {root}")
    elif source == "remote":
        root = _download_and_extract(Path(data_dir))
    else:
        raise ValueError(f"source must be 'local' or 'remote', got {source!r}")

    _reorganize_val(root)
    return root


# ── Data loader helpers ───────────────────────────────────────────────────────

_NORMALIZE = transforms.Normalize(
    mean=[0.4802, 0.4481, 0.3975],
    std=[0.2302, 0.2265, 0.2262],
)
_EVAL_TRANSFORM = transforms.Compose([transforms.ToTensor(), _NORMALIZE])


def _make_loader(dataset, batch_size, sampler=None, shuffle=False):
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )


def get_train_valid_data_loader(
    batch_size,
    augment=True,
    random_seed=1,
    valid_size=0.1,
    shuffle=True,
    source="local",
    local_dir="./dataset-tiny-imagenet-200",
    data_dir="./data",
):
    root = prepare_tiny_imagenet_data(source=source, local_dir=local_dir, data_dir=data_dir)

    if augment:
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(64, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                _NORMALIZE,
            ]
        )
    else:
        train_transform = _EVAL_TRANSFORM

    train_dataset = datasets.ImageFolder(root=root / "train", transform=train_transform)
    valid_dataset = datasets.ImageFolder(root=root / "train", transform=_EVAL_TRANSFORM)

    indices = list(range(len(train_dataset)))
    split = int(np.floor(valid_size * len(train_dataset)))
    if shuffle:
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_loader = _make_loader(
        train_dataset, batch_size, sampler=SubsetRandomSampler(indices[split:])
    )
    valid_loader = _make_loader(
        valid_dataset, batch_size, sampler=SubsetRandomSampler(indices[:split])
    )
    return train_loader, valid_loader


def get_test_data_loader(
    batch_size,
    shuffle=False,
    source="local",
    local_dir="./dataset-tiny-imagenet-200",
    data_dir="./data",
):
    root = prepare_tiny_imagenet_data(source=source, local_dir=local_dir, data_dir=data_dir)
    dataset = datasets.ImageFolder(
        root=root / "val_by_class", transform=_EVAL_TRANSFORM
    )
    return _make_loader(dataset, batch_size, shuffle=shuffle)

ModuleNotFoundError: No module named 'tiny_imagenet'

In [ ]:
# source="local"  → 使用本地已解压的数据集（默认）
# source="remote" → 从 Stanford 下载并解压（适用于 Colab 等无本地数据的环境）
SOURCE = "local"
LOCAL_DIR = "./dataset-tiny-imagenet-200"

num_classes = 200
num_epochs = 30

train_data_loader, valid_data_loader = get_train_valid_data_loader(
    batch_size=64,
    augment=True,
    random_seed=1,
    source=SOURCE,
    local_dir=LOCAL_DIR,
)

test_data_loader = get_test_data_loader(
    batch_size=64,
    source=SOURCE,
    local_dir=LOCAL_DIR,
)

# PyTorch `nn.Conv2d` Parameter

```python
torch.nn.Conv2d(in_channels, out_channels, kernel_size,
                stride=1, padding=0, dilation=1, groups=1,
                bias=True, padding_mode='zeros')
```

## Core Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `in_channels` | int | Number of channels in the input image (e.g., 3 for RGB, 1 for grayscale) |
| `out_channels` | int | Number of channels produced by the convolution (i.e., number of filters) |
| `kernel_size` | int or tuple | Size of the convolving kernel (e.g., `3` for 3×3, `(3,5)` for 3×5) |

## Optional Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `stride` | int or tuple | `1` | Stride of the convolution, controls how the kernel moves |
| `padding` | int, tuple or str | `0` | Padding added to all four sides of the input |
| `dilation` | int or tuple | `1` | Spacing between kernel elements (dilated/atrous convolution) |
| `groups` | int | `1` | Number of blocked connections from input to output channels |
| `bias` | bool | `True` | If `True`, adds a learnable bias to the output |
| `padding_mode` | str | `'zeros'` | Padding mode: `'zeros'`, `'reflect'`, `'replicate'`, or `'circular'` |

## Visual Explanation

https://excalidraw.com/#json=SdlFjnqOskkfBaO-J81Gg,pq4DrjG7b-0-d2QENDoodw

## Output Size Formula

$$
H_{out} = \left\lfloor \frac{H_{in} + 2 \times \text{padding} - \text{dilation} \times (\text{kernel\_size} - 1) - 1}{\text{stride}} + 1 \right\rfloor
$$

## Usage Examples

```python
import torch.nn as nn

# Basic usage: 3 input channels, 64 output channels, 3×3 kernel
conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3)

# Preserve spatial size (padding = kernel_size // 2)
conv = nn.Conv2d(3, 64, kernel_size=3, padding=1)

# Downsampling: stride=2, halves the spatial dimensions
conv = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)

# Dilated convolution: increase receptive field
conv = nn.Conv2d(64, 64, kernel_size=3, dilation=2, padding=2)
```

## Groups Parameter Explained

The `groups` parameter controls the connection between input and output channels:

- **groups=1** (default): All inputs are convolved to all outputs
- **groups=in_channels**: Each input channel is convolved with its own set of filters (depthwise convolution)
- **groups=n**: Input and output channels are divided into `n` groups. For example, at `groups=2`, the operation becomes equivalent to having two conv layers side by side, each seeing half the input channels and producing half the output channels, and both subsequently concatenated.

```python
# Depthwise convolution (used in MobileNet)
depthwise_conv = nn.Conv2d(64, 64, kernel_size=3, groups=64, padding=1)

# Grouped convolution
grouped_conv = nn.Conv2d(64, 128, kernel_size=3, groups=2, padding=1)
```


## Building AlexNet

- The first step to defining any neural network (whether a CNN or not) in PyTorch is to define a class that inherits `nn.Module` as it contains many of the methods that we will need to utilize
- There are two main steps after that. First is initializing the layers that we are going to use in our CNN inside `__init__`, and the other is to define the sequence in which those layers will process the image. This is defined inside the forward function
- For the architecture itself, we first define the convolutional layers using the `nn.Conv2D` function with the appropriate kernel size and the input/output channels. We also apply max pooling using `nn.MaxPool2D` function. The nice thing about PyTorch is that we can combine the convolutional layer, activation function, and max pooling into one single layer (they will be separately applied, but it helps with organization) using the `nn.Sequential` function
- Then we define the fully connected layers using linear (`nn.Linear`) and dropout (`nn.Dropout`) along with ReLu activation function (`nn.ReLU`) and combining these with the nn.Sequential function
- Finally, our last layer outputs 200 neurons, corresponding to the 200 object classes in Tiny ImageNet.


In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=200, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((2, 2))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 2 * 2, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

## Setting Hyperparameters

Before training, we need to set some hyperparameters, such as the loss function and the optimizer to be used along with batch size, learning rate, and number of epochs.

In [ ]:
num_classes = 200
num_epochs = 30
batch_size = 128
learning_rate = 0.01

model = AlexNet(num_classes=num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate,
    weight_decay=5e-4,
    momentum=0.9,
    nesterov=True,
)

## Training

We are ready to train our model at this point:

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

def fit(model, train_loader, test_loader, criterion, optimizer, num_epochs=30):
    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
    total_step = len(train_loader)

    for epoch in range(num_epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = 100.0 * correct / total
        test_loss, test_acc = evaluate(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"{datetime.now()} - Epoch [{epoch + 1}/{num_epochs}], "
            f"Step [{i + 1}/{total_step}], "
            f"train loss: {train_loss:.4f}, train acc: {train_acc:.2f}%, "
            f"test loss: {test_loss:.4f}, test acc: {test_acc:.2f}%"
        )

    return history
history = fit(model, train_data_loader, test_data_loader, criterion, optimizer, num_epochs)

2026-06-20 09:45:29.250362 - Epoch [1/30], Step [352/352], train loss: 5.2988, train acc: 0.49%, test loss: 5.2984, test acc: 0.50%


KeyboardInterrupt: 

### Backup of Training Log

```
2026-06-04 08:25:55.217853 - Epoch [1/30], Step [1407/1407], train loss: 5.2378, train acc: 0.85%, test loss: 5.1788, test acc: 1.06%
2026-06-04 08:27:08.166916 - Epoch [2/30], Step [1407/1407], train loss: 5.0917, train acc: 1.23%, test loss: 5.0222, test acc: 1.66%
2026-06-04 08:28:22.365607 - Epoch [3/30], Step [1407/1407], train loss: 4.9354, train acc: 2.16%, test loss: 4.7983, test acc: 3.59%
2026-06-04 08:29:36.730033 - Epoch [4/30], Step [1407/1407], train loss: 4.5870, train acc: 4.98%, test loss: 4.6573, test acc: 5.55%
2026-06-04 08:30:50.721809 - Epoch [5/30], Step [1407/1407], train loss: 4.2788, train acc: 7.96%, test loss: 4.2041, test acc: 9.46%
2026-06-04 08:32:03.785241 - Epoch [6/30], Step [1407/1407], train loss: 4.0782, train acc: 10.53%, test loss: 3.9943, test acc: 12.11%
2026-06-04 08:33:18.198008 - Epoch [7/30], Step [1407/1407], train loss: 3.9196, train acc: 12.69%, test loss: 3.8242, test acc: 14.31%
2026-06-04 08:34:31.883697 - Epoch [8/30], Step [1407/1407], train loss: 3.7537, train acc: 15.31%, test loss: 3.6285, test acc: 17.80%
2026-06-04 08:35:45.946237 - Epoch [9/30], Step [1407/1407], train loss: 3.6039, train acc: 17.78%, test loss: 3.4549, test acc: 19.78%
2026-06-04 08:36:59.567783 - Epoch [10/30], Step [1407/1407], train loss: 3.4672, train acc: 20.02%, test loss: 3.3711, test acc: 22.55%
2026-06-04 08:38:14.093226 - Epoch [11/30], Step [1407/1407], train loss: 3.3551, train acc: 22.07%, test loss: 3.2823, test acc: 23.07%
2026-06-04 08:39:27.589664 - Epoch [12/30], Step [1407/1407], train loss: 3.2529, train acc: 24.11%, test loss: 3.1974, test acc: 25.04%
2026-06-04 08:40:41.527564 - Epoch [13/30], Step [1407/1407], train loss: 3.1591, train acc: 25.72%, test loss: 3.0182, test acc: 28.16%
2026-06-04 08:41:53.659072 - Epoch [14/30], Step [1407/1407], train loss: 3.0738, train acc: 27.17%, test loss: 2.9693, test acc: 29.67%
2026-06-04 08:43:06.797935 - Epoch [15/30], Step [1407/1407], train loss: 3.0054, train acc: 28.57%, test loss: 3.0095, test acc: 29.06%
2026-06-04 08:44:21.264221 - Epoch [16/30], Step [1407/1407], train loss: 2.9371, train acc: 29.95%, test loss: 3.0535, test acc: 28.00%
2026-06-04 08:45:33.916282 - Epoch [17/30], Step [1407/1407], train loss: 2.8703, train acc: 31.31%, test loss: 2.8399, test acc: 32.35%
2026-06-04 08:46:46.124112 - Epoch [18/30], Step [1407/1407], train loss: 2.8201, train acc: 32.48%, test loss: 2.8241, test acc: 33.79%
2026-06-04 08:47:57.521998 - Epoch [19/30], Step [1407/1407], train loss: 2.7646, train acc: 33.30%, test loss: 2.6992, test acc: 34.96%
2026-06-04 08:49:11.537213 - Epoch [20/30], Step [1407/1407], train loss: 2.7132, train acc: 34.49%, test loss: 2.6302, test acc: 36.40%
2026-06-04 08:50:24.001268 - Epoch [21/30], Step [1407/1407], train loss: 2.6736, train acc: 35.18%, test loss: 2.6516, test acc: 36.17%
2026-06-04 08:51:37.997256 - Epoch [22/30], Step [1407/1407], train loss: 2.6333, train acc: 36.07%, test loss: 2.5991, test acc: 37.50%
2026-06-04 08:52:51.932080 - Epoch [23/30], Step [1407/1407], train loss: 2.5912, train acc: 37.17%, test loss: 2.5900, test acc: 37.47%
2026-06-04 08:54:06.874339 - Epoch [24/30], Step [1407/1407], train loss: 2.5598, train acc: 37.37%, test loss: 2.6079, test acc: 37.63%
2026-06-04 08:55:24.101845 - Epoch [25/30], Step [1407/1407], train loss: 2.5246, train acc: 38.42%, test loss: 2.6370, test acc: 37.10%
2026-06-04 08:56:35.979240 - Epoch [26/30], Step [1407/1407], train loss: 2.4987, train acc: 38.62%, test loss: 2.5840, test acc: 37.38%
2026-06-04 08:57:49.896060 - Epoch [27/30], Step [1407/1407], train loss: 2.4674, train acc: 39.46%, test loss: 2.4733, test acc: 40.06%
2026-06-04 08:59:02.342505 - Epoch [28/30], Step [1407/1407], train loss: 2.4400, train acc: 39.84%, test loss: 2.5197, test acc: 39.53%
2026-06-04 09:00:16.192606 - Epoch [29/30], Step [1407/1407], train loss: 2.4172, train acc: 40.58%, test loss: 2.5225, test acc: 40.13%
2026-06-04 09:01:28.841261 - Epoch [30/30], Step [1407/1407], train loss: 2.3912, train acc: 40.82%, test loss: 2.6289, test acc: 38.09%
```

## Test

Now, we see how our model performs on unseen data:

In [ ]:
model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    total_loss = 0.0

    for images, labels in test_data_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        total_loss += loss.item() * labels.size(0)

        del images, labels, outputs

    print(
        "Accuracy of the network on the {} test images: {:.2f} %".format(
            total,
            100 * correct / total,
        )
    )
    print("Test loss: {:.4f}".format(total_loss / total))

Accuracy of the network on the 10000 test images: 40.86 %
Test loss: 2.4498


## Resources

- [Writing AlexNet from Scratch in PyTorch](https://blog.paperspace.com/alexnet-pytorch/#training)
- [AlexNet | PyTorch](https://pytorch.org/hub/pytorch_vision_alexnet/)
